# The 1-meter Swedish Solar Telescope — 핵심 물리 계산 구현 / Core Physics Implementation

**논문 / Paper**: Scharmer et al., "The 1-meter Swedish solar telescope," *Proc. SPIE*, Vol. 4853, 2003.

**구성 / Contents:**
1. 세 망원경의 종합 비교 / Three Telescopes Compared (McMath, Dunn, SST)
2. Strehl Ratio와 Wavefront Error / Strehl vs Wavefront Error
3. AO 보정 모드 수와 잔차 / AO Correction Modes and Residual Variance
4. Zernike 모드 시각화 / Zernike Mode Visualization
5. Schupmann Corrector: 대기 분산 보상 / Atmospheric Dispersion Compensation
6. 태양 망원경의 진화 요약 / Solar Telescope Evolution Summary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize

plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 12, "axes.grid": True, "grid.alpha": 0.3})

## Part 1: 세 망원경의 종합 비교 / Three Telescopes Compared

Papers #1–#3의 세 망원경을 모든 주요 사양에서 비교합니다. 60년간의 태양 망원경 설계 진화를 한눈에 봅니다.

Comparing the three telescopes from Papers #1–#3 across all key specifications — 60 years of solar telescope evolution at a glance.

In [ ]:
# Three telescopes comparison — radar/spider chart
categories = ["Aperture\n(cm)", "Diffraction\nlimit (\")", "Best actual\nresolution (\")",
              "Spectral\nrange", "Internal\nseeing\nelimination", "AO\ncapability"]

# Normalize all to 0-1 scale (higher = better)
mcmath = [160/160, 0.084/0.2, 1.0/1.5, 0.7, 0.3, 0.0]  # large aperture but limited
dunn =   [76/160, 0.18/0.2,  0.2/1.5, 0.5, 1.0, 0.0]   # vacuum but no AO originally
sst =    [97/160, 0.14/0.2,  0.1/1.5, 1.0, 1.0, 1.0]   # vacuum + AO + wide λ

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: bar chart comparison of key metrics
metrics = {
    "Aperture (cm)": (160, 76, 97),
    "Focal length (m)": (90, 55, 20.3),
    "f-ratio": (56, 72, 21),
    "Diffraction limit\n(arcsec @ 550nm)": (0.084, 0.18, 0.14),
    "Best actual\nresolution (arcsec)": (1.0, 0.2, 0.1),
    "Strehl ratio\n(design)": (0, 0, 0.98),
}

x = np.arange(len(metrics))
width = 0.25
colors = ["#E74C3C", "#3498DB", "#2ECC71"]
names = ["McMath\n(Pierce 1964)", "Dunn VTT\n(Dunn 1964)", "SST\n(Scharmer 2003)"]

for i, (name, color) in enumerate(zip(names, colors)):
    vals = [v[i] for v in metrics.values()]
    max_vals = [max(v) for v in metrics.values()]
    normalized = [v / m if m > 0 else 0 for v, m in zip(vals, max_vals)]
    bars = ax1.bar(x + i * width, normalized, width, label=name, color=color, alpha=0.8)
    # Add actual values
    for j, (bar, val) in enumerate(zip(bars, vals)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val}", ha="center", va="bottom", fontsize=7, fontweight="bold")

ax1.set_xticks(x + width)
ax1.set_xticklabels(metrics.keys(), fontsize=9)
ax1.set_ylabel("Normalized Value")
ax1.set_title("Three Solar Telescopes — Key Metrics\n(normalized to maximum)")
ax1.legend(fontsize=10)
ax1.set_ylim(0, 1.4)

# Right: resolution gap analysis
telescopes = {
    "McMath (160 cm)": {"D": 160, "best": 1.0, "color": "#E74C3C"},
    "Dunn VTT (76 cm)": {"D": 76, "best": 0.2, "color": "#3498DB"},
    "SST (97 cm)": {"D": 97, "best": 0.1, "color": "#2ECC71"},
    "DKIST (400 cm)": {"D": 400, "best": 0.03, "color": "#9B59B6"},
}

for name, info in telescopes.items():
    diff_limit = 1.22 * 550e-7 / info["D"] * 206265
    ax2.barh(name, info["best"], color=info["color"], alpha=0.4, label=f"Actual: {info['best']}\"")
    ax2.barh(name, diff_limit, color=info["color"], alpha=0.9, height=0.4)
    ax2.text(info["best"] + 0.02, name, f"actual: {info['best']}\"", va="center", fontsize=9)
    ax2.text(diff_limit + 0.005, name, f"diff: {diff_limit:.3f}\"", va="bottom", fontsize=8, color="gray")

ax2.set_xlabel("Angular Resolution (arcsec)")
ax2.set_title("Diffraction Limit vs Actual Resolution\n(dark bar = diffraction limit, light bar = actual best)")
ax2.set_xlim(0, 1.3)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

# Print the key insight
print("=" * 60)
print("The Resolution Gap — Why AO Matters")
print("=" * 60)
for name, info in telescopes.items():
    dl = 1.22 * 550e-7 / info["D"] * 206265
    ratio = info["best"] / dl
    print(f"{name:<22} diffraction: {dl:.3f}\"  actual: {info['best']:.3f}\"  gap: {ratio:.1f}×")
print(f"\n→ SST closed the gap to just {0.1/0.14:.1f}× — near diffraction-limited!")
print(f"→ McMath had a gap of {1.0/0.084:.0f}× — completely seeing-limited")

## Part 2: Strehl Ratio와 Wavefront Error / Strehl vs Wavefront Error

Maréchal 근사: $S \approx e^{-(2\pi\sigma/\lambda)^2}$. SST의 Table 1 데이터를 이 관계와 함께 시각화합니다.

In [ ]:
def marechal_strehl(sigma_over_lambda):
    """Maréchal approximation: S = exp(-(2*pi*sigma/lambda)^2)."""
    return np.exp(-(2 * np.pi * sigma_over_lambda)**2)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Strehl vs wavefront error
sigma_ratio = np.linspace(0, 0.3, 300)
strehl = marechal_strehl(sigma_ratio)

ax1.plot(sigma_ratio, strehl, "b-", linewidth=2.5, label="Maréchal: $S = e^{-(2\\pi\\sigma/\\lambda)^2}$")
ax1.axhline(y=0.8, color="red", linestyle="--", alpha=0.7, label="Diffraction limit (S=0.8)")
ax1.axvline(x=1/14, color="red", linestyle=":", alpha=0.3)
ax1.annotate("λ/14", xy=(1/14, 0.8), textcoords="offset points", xytext=(10, -20),
             fontsize=11, color="red")

# Mark SST data points from Table 1
sst_data = {330: 0.82, 350: 0.87, 400: 0.97, 550: 0.98, 800: 0.97, 1100: 0.97}
for lam, S in sst_data.items():
    # Invert Maréchal to get sigma/lambda
    sigma_lam = np.sqrt(-np.log(S)) / (2 * np.pi)
    ax1.plot(sigma_lam, S, "r*", markersize=12, zorder=5)
    ax1.annotate(f"{lam}nm", xy=(sigma_lam, S), textcoords="offset points",
                 xytext=(8, 5), fontsize=8)

ax1.set_xlabel("Wavefront RMS Error (σ/λ)")
ax1.set_ylabel("Strehl Ratio")
ax1.set_title("Strehl Ratio vs Wavefront Error\nwith SST Table 1 data (★)")
ax1.legend(fontsize=10)
ax1.set_ylim(0, 1.05)

# Right: SST Strehl ratio across wavelengths
wavelengths = list(sst_data.keys())
strehls = list(sst_data.values())

ax2.plot(wavelengths, strehls, "go-", markersize=10, linewidth=2, label="SST design Strehl")
ax2.axhline(y=0.8, color="red", linestyle="--", alpha=0.7, label="Diffraction limit threshold")
ax2.fill_between([300, 1200], 0.8, 1.0, alpha=0.05, color="green")
ax2.annotate("ALL wavelengths\nabove diffraction limit!",
             xy=(700, 0.85), fontsize=12, color="green", fontweight="bold",
             bbox=dict(boxstyle="round", facecolor="honeydew"))

ax2.set_xlabel("Wavelength (nm)")
ax2.set_ylabel("Strehl Ratio")
ax2.set_title("SST Strehl Ratio vs Wavelength\n(from Table 1 — optics only, no atmosphere)")
ax2.set_xlim(300, 1200)
ax2.set_ylim(0.7, 1.02)
ax2.legend()

plt.tight_layout()
plt.show()

# Print wavefront errors
print("=" * 55)
print("SST Wavefront Error Analysis (from Table 1)")
print("=" * 55)
print(f"{'λ (nm)':>8} {'Strehl':>8} {'σ/λ':>10} {'σ (nm)':>10} {'σ as λ/N':>10}")
print("-" * 50)
for lam, S in sst_data.items():
    sig_ratio = np.sqrt(-np.log(S)) / (2 * np.pi)
    sig_nm = sig_ratio * lam
    N = 1 / sig_ratio if sig_ratio > 0 else float("inf")
    print(f"{lam:>8} {S:>8.2f} {sig_ratio:>10.4f} {sig_nm:>10.1f} {'λ/'+str(int(N)):>10}")

## Part 3: AO 보정 모드 수와 잔차 / AO Correction Modes and Residual Variance

Kolmogorov 난류에서 처음 $N$개 Zernike 모드 보정 후 잔차:

$$\sigma_N^2 \approx \Delta_N \left(\frac{D}{r_0}\right)^{5/3}$$

여기서 $\Delta_N$은 Noll(1976)의 계수입니다. 이를 통해 SST에서 AO 보정 모드 수에 따른 Strehl을 계산합니다.

In [ ]:
# Noll (1976) residual coefficients for Kolmogorov turbulence
# Delta_N values: residual variance after correcting first N Zernike modes
# sigma^2 = Delta_N * (D/r0)^(5/3)  [rad^2]
noll_coefficients = {
    0: 1.0299,    # no correction
    1: 1.0299,    # piston only (irrelevant)
    2: 0.582,     # tip removed
    3: 0.134,     # tip-tilt removed
    4: 0.111,     # + defocus
    5: 0.0880,    # + astigmatism
    6: 0.0648,    # + astigmatism
    10: 0.0401,   # 10 modes
    15: 0.0279,   # 15 modes
    20: 0.0208,   # 20 modes
    35: 0.0126,   # 35 modes (SST 37-electrode)
    50: 0.0094,   # 50 modes
    100: 0.0050,  # 100 modes
}

# Approximate: Delta_N ≈ 0.2944 * N^(-sqrt(3)/2) for large N
N_range = np.arange(1, 200)
delta_approx = 0.2944 * N_range**(-np.sqrt(3)/2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: residual variance vs number of corrected modes for different D/r0
D_r0_values = {"D/r₀ = 5\n(SST, good seeing)": 5,
               "D/r₀ = 10\n(SST, moderate)": 10,
               "D/r₀ = 20\n(SST, poor seeing)": 20}

for label, D_r0 in D_r0_values.items():
    residual = delta_approx * D_r0**(5/3)
    strehl = np.exp(-residual)
    strehl = np.clip(strehl, 0, 1)
    ax1.plot(N_range, strehl, linewidth=2, label=label)

ax1.axhline(y=0.8, color="red", linestyle="--", alpha=0.7, label="Diffraction limit (S=0.8)")

# Mark SST systems
ax1.axvline(x=10, color="gray", linestyle=":", alpha=0.3)
ax1.annotate("SST 19-electrode\n(~10 modes)", xy=(10, 0.05), fontsize=9, rotation=90, color="gray")
ax1.axvline(x=35, color="gray", linestyle=":", alpha=0.3)
ax1.annotate("SST 37-electrode\n(~35 modes)", xy=(35, 0.05), fontsize=9, rotation=90, color="gray")

ax1.set_xlabel("Number of Corrected Zernike Modes (N)")
ax1.set_ylabel("Strehl Ratio")
ax1.set_title("AO Performance: Strehl vs Corrected Modes\nfor SST (D=97cm) at different seeing")
ax1.set_xscale("log")
ax1.legend(fontsize=9)
ax1.set_xlim(1, 200)
ax1.set_ylim(0, 1.05)

# Right: required modes for diffraction limit at different D/r0
D_r0_range = np.linspace(2, 30, 100)
# For Strehl = 0.8: sigma^2 = -ln(0.8) ≈ 0.223
# Delta_N * (D/r0)^(5/3) = 0.223
# N = (0.2944 * (D/r0)^(5/3) / 0.223)^(2/sqrt(3))
target_sigma2 = -np.log(0.8)
N_required = (0.2944 * D_r0_range**(5/3) / target_sigma2)**(2/np.sqrt(3))

ax2.plot(D_r0_range, N_required, "b-", linewidth=2.5)
ax2.fill_between(D_r0_range, N_required, 2000, alpha=0.05, color="green")
ax2.fill_between(D_r0_range, 0, N_required, alpha=0.05, color="red")

# Mark SST operating points
ax2.plot(5, 10, "r*", markersize=15, zorder=5)
ax2.annotate("SST good seeing\n19-electrode: OK!", xy=(5, 10),
             textcoords="offset points", xytext=(20, 20), fontsize=10,
             color="green", fontweight="bold",
             arrowprops=dict(arrowstyle="->", color="green"))

ax2.plot(20, 35, "r*", markersize=15, zorder=5)
ax2.annotate("SST poor seeing\n37-electrode: NOT enough",
             xy=(20, 35), textcoords="offset points", xytext=(20, -30),
             fontsize=10, color="red", fontweight="bold",
             arrowprops=dict(arrowstyle="->", color="red"))

ax2.set_xlabel("D/r₀ (turbulence strength)")
ax2.set_ylabel("Modes Required for Strehl ≥ 0.8")
ax2.set_title("Required AO Modes for Diffraction Limit\nScharmer's insight: site quality matters!")
ax2.set_yscale("log")
ax2.set_ylim(1, 2000)
ax2.text(15, 500, "Diffraction-limited\nregion ↑", fontsize=11, color="green", ha="center")
ax2.text(15, 3, "Seeing-limited ↓", fontsize=11, color="red", ha="center")

plt.tight_layout()
plt.show()

## Part 4: Zernike 모드 시각화 / Zernike Mode Visualization

처음 15개 Zernike 모드의 wavefront 패턴을 시각화합니다. 각 모드가 어떤 기하학적 왜곡에 대응하는지 직관적으로 이해할 수 있습니다.

In [ ]:
def zernike(n, m, rho, theta):
    """Compute Zernike polynomial Z_n^m on polar coordinates (rho, theta).
    
    Uses the Noll single-index convention for the first ~15 modes.
    """
    if n == 0 and m == 0: return np.ones_like(rho)                     # Z1: Piston
    if n == 1 and m == 1: return 2 * rho * np.cos(theta)               # Z2: Tip
    if n == 1 and m == -1: return 2 * rho * np.sin(theta)              # Z3: Tilt
    if n == 2 and m == 0: return np.sqrt(3) * (2*rho**2 - 1)           # Z4: Defocus
    if n == 2 and m == -2: return np.sqrt(6) * rho**2 * np.sin(2*theta)  # Z5: Astig 45°
    if n == 2 and m == 2: return np.sqrt(6) * rho**2 * np.cos(2*theta)   # Z6: Astig 0°
    if n == 3 and m == -1: return np.sqrt(8) * (3*rho**3 - 2*rho) * np.sin(theta)  # Z7: Coma y
    if n == 3 and m == 1: return np.sqrt(8) * (3*rho**3 - 2*rho) * np.cos(theta)   # Z8: Coma x
    if n == 3 and m == -3: return np.sqrt(8) * rho**3 * np.sin(3*theta)  # Z9: Trefoil
    if n == 3 and m == 3: return np.sqrt(8) * rho**3 * np.cos(3*theta)   # Z10: Trefoil
    if n == 4 and m == 0: return np.sqrt(5) * (6*rho**4 - 6*rho**2 + 1)  # Z11: Spherical
    if n == 4 and m == 2: return np.sqrt(10) * (4*rho**4 - 3*rho**2) * np.cos(2*theta)  # Z12
    if n == 4 and m == -2: return np.sqrt(10) * (4*rho**4 - 3*rho**2) * np.sin(2*theta)  # Z13
    if n == 4 and m == 4: return np.sqrt(10) * rho**4 * np.cos(4*theta)  # Z14
    if n == 4 and m == -4: return np.sqrt(10) * rho**4 * np.sin(4*theta)  # Z15
    return np.zeros_like(rho)


# Zernike modes to display (Noll index, n, m, name)
modes = [
    (1, 0, 0, "Z₁ Piston"),
    (2, 1, 1, "Z₂ Tip (x)"),
    (3, 1, -1, "Z₃ Tilt (y)"),
    (4, 2, 0, "Z₄ Defocus"),
    (5, 2, -2, "Z₅ Astig 45°"),
    (6, 2, 2, "Z₆ Astig 0°"),
    (7, 3, -1, "Z₇ Coma y"),
    (8, 3, 1, "Z₈ Coma x"),
    (9, 3, -3, "Z₉ Trefoil"),
    (10, 3, 3, "Z₁₀ Trefoil"),
    (11, 4, 0, "Z₁₁ Spherical"),
    (12, 4, 2, "Z₁₂ 2nd Astig"),
    (13, 4, -2, "Z₁₃ 2nd Astig"),
    (14, 4, 4, "Z₁₄ Quadrafoil"),
    (15, 4, -4, "Z₁₅ Quadrafoil"),
]

# Create grid
size = 200
x = np.linspace(-1, 1, size)
y = np.linspace(-1, 1, size)
X, Y = np.meshgrid(x, y)
rho = np.sqrt(X**2 + Y**2)
theta = np.arctan2(Y, X)
mask = rho <= 1.0

fig, axes = plt.subplots(3, 5, figsize=(18, 11))

for idx, (noll, n, m, name) in enumerate(modes):
    ax = axes[idx // 5, idx % 5]
    Z = zernike(n, m, rho, theta)
    Z[~mask] = np.nan
    im = ax.imshow(Z, cmap="RdBu_r", extent=[-1, 1, -1, 1], vmin=-2, vmax=2)
    ax.set_title(name, fontsize=10, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])
    # Draw unit circle
    circle = plt.Circle((0, 0), 1, fill=False, color="black", linewidth=1.5)
    ax.add_patch(circle)
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")

plt.suptitle("Zernike Polynomials — First 15 Modes\n"
             "Red = wavefront advanced, Blue = wavefront retarded\n"
             "AO corrects these modes sequentially to improve image quality",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Atmospheric contribution per mode
print("=" * 50)
print("Atmospheric Turbulence Contribution per Mode")
print("(Kolmogorov turbulence, Noll 1976)")
print("=" * 50)
contributions = [
    ("Z₁ Piston", "—", "irrelevant"),
    ("Z₂+Z₃ Tip-Tilt", "~87%", "image jitter"),
    ("Z₄ Defocus", "~5%", "focus shift"),
    ("Z₅+Z₆ Astigmatism", "~3%", "elongation"),
    ("Z₇+Z₈ Coma", "~2%", "comet tail"),
    ("Z₉+Z₁₀ Trefoil", "~1%", "three-fold"),
    ("Z₁₁ Spherical", "<1%", "center-edge focus"),
    ("Z₁₂+ Higher", "<1% each", "fine structure"),
]
for name, pct, desc in contributions:
    print(f"  {name:<25} {pct:>8}  — {desc}")

## Part 5: Schupmann Corrector — 대기 분산 보상 / Atmospheric Dispersion Compensation

대기 분산과 Schupmann corrector의 보상 효과를 Table 2 데이터로 시각화합니다.

In [ ]:
# Table 2 data: atmospheric dispersion at 15° elevation
table2 = {
    "350–1100 nm": {"atm": 7.0, "residual": 0.46, "pupil_mm": 3.0, "improvement": 15},
    "400–900 nm":  {"atm": 5.0, "residual": 0.2,  "pupil_mm": 3.0, "improvement": 24},
    "350–650 nm":  {"atm": 5.6, "residual": 0.1,  "pupil_mm": 3.2, "improvement": 55},
    "700–1100 nm": {"atm": 1.1, "residual": 0.11, "pupil_mm": 2.3, "improvement": 10},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: before and after Schupmann correction
bands = list(table2.keys())
atm_vals = [table2[b]["atm"] for b in bands]
res_vals = [table2[b]["residual"] for b in bands]

x = np.arange(len(bands))
width = 0.35
bars1 = ax1.bar(x - width/2, atm_vals, width, label="Atmospheric dispersion",
                color="coral", alpha=0.8, edgecolor="black")
bars2 = ax1.bar(x + width/2, res_vals, width, label="After Schupmann correction",
                color="steelblue", alpha=0.8, edgecolor="black")

# Add improvement factors
for i, b in enumerate(bands):
    imp = table2[b]["improvement"]
    ax1.text(i, max(atm_vals[i], res_vals[i]) + 0.3,
             f"{imp}×", ha="center", fontsize=11, fontweight="bold", color="green")

ax1.set_xticks(x)
ax1.set_xticklabels(bands, fontsize=9)
ax1.set_ylabel("Dispersion (arcsec)")
ax1.set_title("Atmospheric Dispersion at 15° Elevation\nBefore and After Schupmann Correction")
ax1.legend()

# Add SST resolution line
ax1.axhline(y=0.14, color="green", linestyle="--", alpha=0.5)
ax1.annotate("SST diffraction limit (0.14\")", xy=(2, 0.14),
             textcoords="offset points", xytext=(0, 8), fontsize=9, color="green")

# Right: improvement factor vs wavelength range center
centers = [725, 650, 500, 900]  # approximate center wavelengths
improvements = [15, 24, 55, 10]

# Model: atmospheric dispersion vs elevation
elevations = np.linspace(10, 90, 100)
# Atmospheric dispersion scales approximately as tan(z) where z = 90° - elevation
# For 400-900nm band: ~5" at 15° elevation
disp_400_900 = 5.0 * np.tan(np.radians(90 - 15)) / np.tan(np.radians(90 - elevations))
disp_400_900_corrected = disp_400_900 / 24  # Schupmann improvement factor

ax2.plot(elevations, disp_400_900, "r-", linewidth=2, label="Atmospheric (400–900 nm)")
ax2.plot(elevations, disp_400_900_corrected, "b-", linewidth=2,
         label="After Schupmann (400–900 nm)")
ax2.axhline(y=0.14, color="green", linestyle="--", alpha=0.5, label="SST diffraction limit")

ax2.fill_between(elevations, 0, 0.14, alpha=0.05, color="green")
ax2.annotate("Usable for diffraction-limited\nimaging (below 0.14\")",
             xy=(50, 0.07), fontsize=10, color="green",
             bbox=dict(boxstyle="round", facecolor="honeydew"))

ax2.set_xlabel("Elevation (degrees)")
ax2.set_ylabel("Dispersion (arcsec)")
ax2.set_title("Atmospheric Dispersion vs Elevation\n(400–900 nm band)")
ax2.legend()
ax2.set_ylim(0, 3)

plt.tight_layout()
plt.show()

## Part 6: 태양 망원경의 진화 요약 / Solar Telescope Evolution Summary

Papers #1–#3을 통해 본 60년간의 설계 패러다임 변화를 정리합니다.

In [ ]:
# Evolution of solar telescope design — timeline visualization
fig, ax = plt.subplots(figsize=(16, 8))

# Timeline data
telescopes = [
    (1964, "McMath\n(Pierce)", 160, 1.0, "#E74C3C",
     "Water-cooled\nenclosure\nNo AO"),
    (1964, "Dunn VTT\n(Dunn)", 76, 0.2, "#3498DB",
     "VACUUM\nTurret\nNo AO"),
    (2003, "SST\n(Scharmer)", 97, 0.1, "#2ECC71",
     "Vacuum\nSinglet+Schupmann\nAO integrated"),
    (2020, "DKIST", 400, 0.03, "#9B59B6",
     "Active cooling\n(no vacuum)\n1600-act AO"),
]

for year, name, aperture, resolution, color, desc in telescopes:
    # Bubble size proportional to aperture
    ax.scatter(year, resolution, s=aperture*3, color=color, alpha=0.6,
               edgecolors="black", linewidth=1.5, zorder=5)
    ax.annotate(f"{name}\nD={aperture}cm\n{resolution}\"",
                xy=(year, resolution),
                textcoords="offset points",
                xytext=(30 if year < 2010 else -80, 20 if resolution > 0.15 else -40),
                fontsize=10, fontweight="bold", color=color,
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5),
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# Innovation labels
innovations = [
    (1964, 0.6, "Paradigm 1:\n'More light'\n(bigger aperture)", "#E74C3C"),
    (1964, 0.15, "Paradigm 2:\n'Remove air'\n(vacuum)", "#3498DB"),
    (2003, 0.08, "Paradigm 3:\n'Correct atmosphere'\n(AO)", "#2ECC71"),
]
for year, y, text, color in innovations:
    ax.text(year + 1, y, text, fontsize=10, color=color, style="italic",
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.7))

# Diffraction limit curve for different apertures
years_range = np.linspace(1960, 2025, 100)
ax.axhline(y=0.14, color="green", linestyle=":", alpha=0.3)
ax.text(2022, 0.145, "SST diff. limit", fontsize=8, color="green")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Best Achieved Resolution (arcsec)", fontsize=12)
ax.set_title("60 Years of Solar Telescope Evolution\n"
             "Three Paradigms: Light → Vacuum → AO\n"
             "(bubble size ∝ aperture)", fontsize=14, fontweight="bold")
ax.set_yscale("log")
ax.set_ylim(0.02, 2)
ax.set_xlim(1960, 2025)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

# Print the three paradigm shifts
print("=" * 65)
print("Three Paradigms of Solar Telescope Design")
print("=" * 65)
print()
print("Paradigm 1 — McMath (1964): 'Collect more light'")
print("  → 160 cm aperture, water-cooled enclosure")
print("  → Internal seeing reduced but NOT eliminated")
print("  → Best: ~1.0\" (12× worse than diffraction limit)")
print()
print("Paradigm 2 — Dunn (1964): 'Remove the air'")
print("  → 76 cm aperture, full vacuum")
print("  → Internal seeing = 0")
print("  → Best: ~0.2\" (close to diffraction limit)")
print()
print("Paradigm 3 — SST (2003): 'Correct the atmosphere'")
print("  → 97 cm aperture, vacuum + singlet + Schupmann + AO")
print("  → Internal seeing = 0, external seeing corrected by AO")
print("  → Best: 0.1\" (AT the diffraction limit!)")
print()
print("Each paradigm addressed the NEXT limiting factor:")
print("  Light → Internal turbulence → External turbulence → ???")
print("  (Next: DKIST at 4m — will MCAO extend this further?)")

## 요약 / Summary

| Part | 주제 / Topic | 핵심 결과 / Key Result |
|------|------------|---------------------|
| 1 | Three Telescopes | McMath: 12× gap, Dunn: 1.1×, SST: 0.7× (diffraction-limited!) |
| 2 | Strehl Ratio | SST Strehl ≥ 0.82 across 330–1100 nm; 0.98 at 550 nm |
| 3 | AO Modes | Good seeing (D/r₀=5): 10 modes suffice; Poor seeing (D/r₀=20): 35 not enough |
| 4 | Zernike Modes | Tip-tilt = 87% of turbulence; first 15 modes visualized |
| 5 | Atmospheric Dispersion | Schupmann reduces dispersion 15–55× (free bonus!) |
| 6 | Evolution | Three paradigms: Light (McMath) → Vacuum (Dunn) → AO (SST) |

**다음 논문 / Next Paper**: #4 Goode & Cao (2012) — New Solar Telescope (NST)
→ 1.6m off-axis 설계로 입사창 없이 진공 문제를 다른 방식으로 해결
→ 1.6-m off-axis design — solves vacuum problem differently without entrance window